In [9]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# dx = 1 km; Np = 1M; Nt = 5 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
res='1km'
Np_str='1e6'

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'


# dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

In [10]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [12]:
#DOMAIN SUBSETTING
############################################################
ocean_percent=2/8

left_to_coast=data['xh'][0]+(data['xh'][-1]-data['xh'][0])*ocean_percent
where_coast_xh=np.where(data['xh']>=left_to_coast)[0][0]#-25
where_coast_xf=np.where(data['xf']>=left_to_coast)[0][0]#-25
end_xh=len(data['xh'])-1-50
end_xf=len(data['xf'])-1-50

print(f'x in {0}:{where_coast_xh-1} FOR SEA')
print(f'x in {where_coast_xh}:{end_xh} FOR LAND')
# t_end=78 
# if res=='250m':t_end=410
# print(f't in {0}:{t_end} (6.5 hours)')
if t_res=='5min':
    t_start=36 # 9:00 am (3 hours in)
if t_res=='1min':
    t_start=36*5 # 9:00 am (3 hours in)
print(f't in {t_start}:end (8 hours)')


#SUBSETTING CODE
data=data.isel(time=slice(t_start,None),xh=slice(where_coast_xh,end_xh+1),xf=slice(where_coast_xf,end_xf+1))
parcel=parcel.isel(time=slice(t_start,None))

x in 0:83 FOR SEA
x in 84:283 FOR LAND
t in 180:end (8 hours)


In [13]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=180 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(data['time']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=1
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 0, end_job = 3


In [14]:
#Indexing Array with JobArray
data=data.isel(time=slice(start_job,end_job))
parcel=parcel.isel(time=slice(start_job,end_job))
#(for 150_000_000 parcels use 500-1000 jobs)

In [15]:
def call_variables(): 
    variable='winterp'; w_data=data[variable].data #get w_data and interpolation w data z coordinate from zh to zf
    variable='qv'; qv_data=data[variable].data # get qc data
    variable='qc'; qc_data=data[variable].data # get qc data
    variable='qi'; qi_data=data[variable].data # get qc data
    qc_plus_qi=qc_data+qi_data

    return w_data,qv_data,qc_data,qi_data,qc_plus_qi

print('calling variables')
[w_data,qv_data,qc_data,qi_data,qc_plus_qi]=call_variables()
print('done')

calling variables
done


In [16]:
################################
#RUNNING

In [17]:
def call_variables(): #***
    # variable='winterp'; w_data=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='wb_hadv'; wb_hadv=data[variable].interp(zf=data['zh']).data #'w budget: horizontal advection (non-diff component)'
    variable='wb_vadv'; wb_vadv=data[variable].interp(zf=data['zh']).data #'w budget: vertical advection (non-diff component)'
    variable='wb_hidiff'; wb_hidiff=data[variable].interp(zf=data['zh']).data #'w budget: horiz implicit diffusion'
    variable='wb_vidiff'; wb_vidiff=data[variable].interp(zf=data['zh']).data #'w budget: vert implicit diffusion'
    variable='wb_hturb'; wb_hturb=data[variable].interp(zf=data['zh']).data #'w budget: horizontal parameterized turbulence'
    variable='wb_vturb'; wb_vturb=data[variable].interp(zf=data['zh']).data #'w budget: vertical parameterized turbulence'
    variable='wb_pgrad'; wb_pgrad=data[variable].interp(zf=data['zh']).data #'w budget: pressure gradient'
    variable='wb_buoy'; wb_bouy=data[variable].interp(zf=data['zh']).data #'w budget: buoyancy'

    return wb_hadv,wb_vadv,wb_hidiff,wb_vidiff,wb_hturb,wb_vturb,wb_pgrad,wb_bouy

print('calling variables')
[wb_hadv,wb_vadv,wb_hidiff,wb_vidiff,wb_hturb,wb_vturb,wb_pgrad,wb_buoy]=call_variables()
print('done')

calling variables
done


In [18]:
def call_variables(): #***
    # variable='qv'; qv_data=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='qvb_hadv'; qvb_hadv=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='qvb_vadv'; qvb_vadv=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='qvb_hidiff'; qvb_hidiff=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='qvb_vidiff'; qvb_vidiff=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='qvb_hturb'; qvb_hturb=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='qvb_vturb'; qvb_vturb=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='qvb_mp'; qvb_mp=data[variable].data #'w budget: horizontal advection (non-diff component)'


    return qvb_hadv,qvb_vadv,qvb_hidiff,qvb_vidiff,qvb_hturb,qvb_vturb,qvb_mp

print('calling variables')
[qvb_hadv,qvb_vadv,qvb_hidiff,qvb_vidiff,qvb_hturb,qvb_vturb,qvb_mp]=call_variables()
print('done')

calling variables
done


In [19]:
def call_variables(): #***
    variable='th'; th_data=data[variable].data #'w budget: horizontal advection (non-diff component)'    
    variable='ptb_hadv'; ptb_hadv=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='ptb_vadv'; ptb_vadv=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='ptb_hidiff'; ptb_hidiff=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='ptb_vidiff'; ptb_vidiff=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='ptb_hturb'; ptb_hturb=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='ptb_vturb'; ptb_vturb=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='ptb_mp'; ptb_mp=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='ptb_rad'; ptb_rad=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='ptb_div'; ptb_div=data[variable].data #'w budget: horizontal advection (non-diff component)'
    variable='ptb_diss'; ptb_diss=data[variable].data #'w budget: horizontal advection (non-diff component)'


    return th_data,ptb_hadv,ptb_vadv,ptb_hidiff,ptb_vidiff,ptb_hturb,ptb_vturb,ptb_mp,ptb_rad,ptb_div,ptb_diss

print('calling variables')
[th_data,ptb_hadv,ptb_vadv,ptb_hidiff,ptb_vidiff,ptb_hturb,ptb_vturb,ptb_mp,ptb_rad,ptb_div,ptb_diss]=call_variables()
print('done')

calling variables
done


In [21]:
check_memory(globals())

Top 10 objects with highest memory usage
{'qc_plus_qi': '16.32 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'FuncAnimation': '0.0 MB', 'PillowWriter': '0.0 MB', 'input_file': '0.0 MB', 'axs': '0.0 MB', 'w_data': '0.0 MB'}

0.01632 GB in use overall


In [22]:
##########################################################################################
#Functions
# Full Profile function makes profile together for all timesteps. AveragedProfiles funciton takes the final mean of the combined profile.

In [28]:
#W Budgets

# Thresholds
w_thresh1 = 0.1
w_thresh2 = 0.5
qcqi_thresh = 1e-6

def final_profile(vars_list, type):
    zhs = data['zh'].values
    profiles = {}  # Store profiles for all variables

    # Initialize profiles for each variable
    for var in vars_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs

    # Threshold mask
    if type == "general":
        where_updraft = (w_data >= w_thresh1) & (qc_plus_qi < qcqi_thresh)
    elif type == "cloudy":
        where_updraft = (w_data >= w_thresh2) & (qc_plus_qi >= qcqi_thresh)
    t_ind, z_ind, y_ind, x_ind = np.where(where_updraft)

    # Variable selection dictionary
    var_data = {
        'w': w_data,
        'wb_hadv': wb_hadv, 'wb_vadv': wb_vadv, 'wb_hidiff': wb_hidiff, 
        'wb_vidiff': wb_hadv, 'wb_hturb': wb_hturb, 'wb_vturb': wb_vturb, 
        'wb_pgrad': wb_pgrad, 'wb_buoy': wb_buoy
    }

    # Iterate over each variable in vars_list and bin the data
    for var in vars_list:
        masked_data = var_data[var][where_updraft]
        np.add.at(profiles[var][:, 0], z_ind, masked_data)
        np.add.at(profiles[var][:, 1], z_ind, 1)

    return profiles

In [29]:
dir2=dir+'Project_Algorithms/Domain_Profiles/'

for type in ['general','cloudy']:
    print(f"working on {type} type\n")
    
    vars_list = ['w',
                 'wb_hadv','wb_vadv','wb_hidiff','wb_vidiff',
                 'wb_hturb','wb_vturb','wb_pgrad','wb_buoy']
    profiles = final_profile(vars_list, type)
    
    #Saving eulerian_profiles
    import h5py
    
    if type == "general":
        output_file = dir2+f'job_out/W_BUDGET_general_eulerian_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5' 
    elif type == "cloudy":
        output_file = dir2+f'job_out/W_BUDGET_cloudy_eulerian_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
    
    with h5py.File(output_file, 'w') as f:
        for var in profiles:
            profile_var = profiles[var]
            f.create_dataset(f'profile_{var}', data=profile_var, compression="gzip")

working on general type

working on cloudy type



In [30]:
#QV Budgets

# Thresholds
w_thresh1 = 0.1
w_thresh2 = 0.5
qcqi_thresh = 1e-6

def final_profile(vars_list, type):
    zhs = data['zh'].values
    profiles = {}  # Store profiles for all variables

    # Initialize profiles for each variable
    for var in vars_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs

    # Threshold mask
    if type == "general":
        where_updraft = (w_data >= w_thresh1) & (qc_plus_qi < qcqi_thresh)
    elif type == "cloudy":
        where_updraft = (w_data >= w_thresh2) & (qc_plus_qi >= qcqi_thresh)
    t_ind, z_ind, y_ind, x_ind = np.where(where_updraft)

    # Variable selection dictionary
    var_data = {
        'qv': qv_data,
        'qvb_hadv': qvb_hadv, 'qvb_vadv': qvb_vadv, 'qvb_hidiff': qvb_hidiff, 
        'qvb_vidiff': qvb_hadv, 'qvb_hturb': qvb_hturb, 'qvb_vturb': qvb_vturb, 
        'qvb_mp': qvb_mp
    }

    # Iterate over each variable in vars_list and bin the data
    for var in vars_list:
        masked_data = var_data[var][where_updraft]
        np.add.at(profiles[var][:, 0], z_ind, masked_data)
        np.add.at(profiles[var][:, 1], z_ind, 1)

    return profiles

In [31]:
dir2=dir+'Project_Algorithms/Domain_Profiles/'

for type in ['general','cloudy']:
    print(f"working on {type} type\n")
    
    vars_list = ['qv',
                 'qvb_hadv','qvb_vadv','qvb_hidiff',
                 'qvb_vidiff','qvb_hturb','qvb_vturb','qvb_mp']
    profiles = final_profile(vars_list, type)
    
    #Saving eulerian_profiles
    import h5py
    
    if type == "general":
        output_file = dir2+f'job_out/QV_BUDGET_general_eulerian_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5' 
    elif type == "cloudy":
        output_file = dir2+f'job_out/QV_BUDGET_cloudy_eulerian_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
    
    with h5py.File(output_file, 'w') as f:
        for var in profiles:
            profile_var = profiles[var]
            f.create_dataset(f'profile_{var}', data=profile_var, compression="gzip")

working on general type

working on cloudy type



In [32]:
#TH Budgets

# Thresholds
w_thresh1 = 0.1
w_thresh2 = 0.5
qcqi_thresh = 1e-6

def final_profile(vars_list, type):
    zhs = data['zh'].values
    profiles = {}  # Store profiles for all variables

    # Initialize profiles for each variable
    for var in vars_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs

    # Threshold mask
    if type == "general":
        where_updraft = (w_data >= w_thresh1) & (qc_plus_qi < qcqi_thresh)
    elif type == "cloudy":
        where_updraft = (w_data >= w_thresh2) & (qc_plus_qi >= qcqi_thresh)
    t_ind, z_ind, y_ind, x_ind = np.where(where_updraft)

    # Variable selection dictionary
    var_data = {
        'th': th_data,
        'ptb_hadv': ptb_hadv, 'ptb_vadv': ptb_vadv, 'ptb_hidiff': ptb_hidiff, 
        'ptb_vidiff': ptb_hadv, 'ptb_hturb': ptb_hturb, 'ptb_vturb': ptb_vturb, 
        'ptb_mp': ptb_mp,'ptb_rad': ptb_rad,'ptb_div': ptb_div,'ptb_diss': ptb_diss
    }



    
    # Iterate over each variable in vars_list and bin the data
    for var in vars_list:
        masked_data = var_data[var][where_updraft]
        np.add.at(profiles[var][:, 0], z_ind, masked_data)
        np.add.at(profiles[var][:, 1], z_ind, 1)

    return profiles

In [33]:
dir2=dir+'Project_Algorithms/Domain_Profiles/'

for type in ['general','cloudy']:
    print(f"working on {type} type\n")
    
    vars_list = ['th','ptb_hadv','ptb_vadv','ptb_hidiff','ptb_vidiff',
                 'ptb_hturb','ptb_vturb','ptb_mp','ptb_rad','ptb_div','ptb_diss']
    profiles = final_profile(vars_list, type)
    
    #Saving eulerian_profiles
    import h5py
    
    if type == "general":
        output_file = dir2+f'job_out/TH_BUDGET_general_eulerian_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5' 
    elif type == "cloudy":
        output_file = dir2+f'job_out/TH_BUDGET_cloudy_eulerian_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
    
    with h5py.File(output_file, 'w') as f:
        for var in profiles:
            profile_var = profiles[var]
            f.create_dataset(f'profile_{var}', data=profile_var, compression="gzip")

working on general type

working on cloudy type



In [ ]:
##########################################################################################

In [34]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOB ARRAY IS RUNNING
# recombine=True

In [38]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Domain_Profiles/'
    
    types=['general','cloudy']
    for type in types:
        #MAKING OUTPUT FILE PATH
        if type == "general":
            output_file = dir2+f'job_out/W_BUDGET_general_eulerian_profiles_{res}_{Np_str}_5min.h5' 
        elif type == "cloudy":
            output_file = dir2+f'job_out/W_BUDGET_cloudy_eulerian_profiles_{res}_{Np_str}_5min.h5'
        
        #MAKING PROFILES DICTIONARY
        zhs = data['zh'].values
        profiles = {}  # Store profiles for all variables
        vars_list = ['w',
                     'wb_hadv','wb_vadv','wb_hidiff','wb_vidiff',
                     'wb_hturb','wb_vturb','wb_pgrad','wb_buoy']
        for var in vars_list:
            profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
            profiles[var][:, 2] = zhs 
        
        num_jobs=180
        for job_id in np.arange(1,num_jobs+1):
            if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
    
            #CALLING IN DATA
            if type == "general":
                input_file = dir2+f'job_out/W_BUDGET_general_eulerian_profiles_{res}_{Np_str}_5min_{job_id}.h5' 
            elif type == "cloudy":
                input_file = dir2+f'job_out/W_BUDGET_cloudy_eulerian_profiles_{res}_{Np_str}_5min_{job_id}.h5'
    
            #COMPILING PROFILES
            with h5py.File(input_file, 'r') as f:
                for var in vars_list:  
                    profiles[var][:,0:1+1]+=f[f'profile_{var}'][:,0:1+1]
        
        #SAVING INTO FINAL FORM
        with h5py.File(output_file, 'w') as f:
            for var in profiles:
                profile_var = profiles[var]
                f.create_dataset(f'profile_{var}', data=profile_var, compression="gzip")

In [39]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Domain_Profiles/'
    
    types=['general','cloudy']
    for type in types:
        #MAKING OUTPUT FILE PATH
        if type == "general":
            output_file = dir2+f'job_out/QV_BUDGET_general_eulerian_profiles_{res}_{Np_str}_5min.h5' 
        elif type == "cloudy":
            output_file = dir2+f'job_out/QV_BUDGET_cloudy_eulerian_profiles_{res}_{Np_str}_5min.h5'
        
        #MAKING PROFILES DICTIONARY
        zhs = data['zh'].values
        profiles = {}  # Store profiles for all variables
        vars_list = ['qv',
                     'qvb_hadv','qvb_vadv','qvb_hidiff',
                     'qvb_vidiff','qvb_hturb','qvb_vturb','qvb_mp']
        for var in vars_list:
            profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
            profiles[var][:, 2] = zhs 
        
        num_jobs=180
        for job_id in np.arange(1,num_jobs+1):
            if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
    
            #CALLING IN DATA
            if type == "general":
                input_file = dir2+f'job_out/QV_BUDGET_general_eulerian_profiles_{res}_{Np_str}_5min_{job_id}.h5' 
            elif type == "cloudy":
                input_file = dir2+f'job_out/QV_BUDGET_cloudy_eulerian_profiles_{res}_{Np_str}_5min_{job_id}.h5'
    
            #COMPILING PROFILES
            with h5py.File(input_file, 'r') as f:
                for var in vars_list:  
                    profiles[var][:,0:1+1]+=f[f'profile_{var}'][:,0:1+1]
        
        #SAVING INTO FINAL FORM
        with h5py.File(output_file, 'w') as f:
            for var in profiles:
                profile_var = profiles[var]
                f.create_dataset(f'profile_{var}', data=profile_var, compression="gzip")

In [40]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Domain_Profiles/'
    
    types=['general','cloudy']
    for type in types:
        #MAKING OUTPUT FILE PATH
        if type == "general":
            output_file = dir2+f'job_out/TH_BUDGET_general_eulerian_profiles_{res}_{Np_str}_5min.h5' 
        elif type == "cloudy":
            output_file = dir2+f'job_out/TH_BUDGET_cloudy_eulerian_profiles_{res}_{Np_str}_5min.h5'
        
        #MAKING PROFILES DICTIONARY
        zhs = data['zh'].values
        profiles = {}  # Store profiles for all variables
        vars_list = ['th',
                     'ptb_hadv','ptb_vadv','ptb_hidiff','ptb_vidiff',
                     'ptb_hturb','ptb_vturb','ptb_mp','ptb_rad','ptb_div','ptb_diss']
        for var in vars_list:
            profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
            profiles[var][:, 2] = zhs 
        
        num_jobs=180
        for job_id in np.arange(1,num_jobs+1):
            if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
    
            #CALLING IN DATA
            if type == "general":
                input_file = dir2+f'job_out/TH_BUDGET_general_eulerian_profiles_{res}_{Np_str}_5min_{job_id}.h5' 
            elif type == "cloudy":
                input_file = dir2+f'job_out/TH_BUDGET_cloudy_eulerian_profiles_{res}_{Np_str}_5min_{job_id}.h5'
    
            #COMPILING PROFILES
            with h5py.File(input_file, 'r') as f:
                for var in vars_list:  
                    profiles[var][:,0:1+1]+=f[f'profile_{var}'][:,0:1+1]
        
        #SAVING INTO FINAL FORM
        with h5py.File(output_file, 'w') as f:
            for var in profiles:
                profile_var = profiles[var]
                f.create_dataset(f'profile_{var}', data=profile_var, compression="gzip")

In [41]:
###########################
#Plotting
plotting2=False #KEEP FALSE WHEN JOB ARRAYS IS RUNNING
# plotting2=True

In [42]:
#thresholds
w_thresh1=0.1
w_thresh2=0.5
qcqi_thresh=1e-6

In [43]:
if plotting2==True:
    #WBUDGET
    label=1
    
    import h5py
    #Produced averaged profiles for plotting
    def averaged_profiles(var):
        profile=globals()[f"profile_{var}"]
    
        out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
        out_var=np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T #divides the data column by the counter column
        return out_var
    
    ####################################################################
    dim='1km'
    # dim='250m'
    label=1
    
    #Uses Plotting Function 
    fig, axs = plt.subplots(2, 2, figsize=(10, 10))
    plt.subplots_adjust(top=0.95)
    
    fig.suptitle(f"Average General (" + r"$w_{thresh}$: w>= " + f"{w_thresh1} & " + r"$(q_c+q_i)_{thresh}$ < " + f"{qcqi_thresh}" + ") vs Cloudy (" + r"$w_{thresh}$: w>= " + f"{w_thresh2} & " + r"$(q_c+q_i)_{thresh}$ >= " + f"{qcqi_thresh}" + ") Updraft WBUDGET Profiles (Eulerian)")
    ax1, ax2, ax3, ax4 = axs.flatten()
    
    ####################################################################
    for type in ["general","cloudy"]:
        print(f'currently on type {type}')
        
        if type == "general":
            input_file = dir2+f'job_out/W_BUDGET_general_eulerian_profiles_{res}_{t_res}_{Np_str}.h5' 
        elif type == "cloudy":
            input_file = dir2+f'job_out/W_BUDGET_cloudy_eulerian_profiles_{res}_{t_res}_{Np_str}.h5' 
        
        with h5py.File(input_file, 'r') as f:
            profile_w = np.array(f['profile_w'])  # w budget: horizontal advection (non-diff component)
            profile_wb_hadv = np.array(f['profile_wb_hadv'])  # w budget: horizontal advection (non-diff component)
            profile_wb_vadv = np.array(f['profile_wb_vadv'])  # w budget: vertical advection (non-diff component)
            profile_wb_hidiff = np.array(f['profile_wb_hidiff'])  # w budget: horiz implicit diffusion
            profile_wb_vidiff = np.array(f['profile_wb_vidiff'])  # w budget: vert implicit diffusion
            profile_wb_hturb = np.array(f['profile_wb_hturb'])  # w budget: horizontal parameterized turbulence
            profile_wb_vturb = np.array(f['profile_wb_vturb'])  # w budget: vertical parameterized turbulence
            profile_wb_pgrad = np.array(f['profile_wb_pgrad'])  # w budget: pressure gradient
            profile_wb_buoy = np.array(f['profile_wb_buoy'])  # w budget: buoyancy
            
    
    
        #Plotting function
        def plotting(out_var,axis,label,color):
            axis.plot(out_var[:,0],out_var[:,1],label=label,color=color)
            axis.grid(True)
        
        #Uses Averaged_Profiles Function
        vars = [
        'w',
        'wb_hadv',
        'wb_vadv',
        'wb_hidiff',
        'wb_vidiff',
        'wb_hturb',
        'wb_vturb',
        'wb_pgrad',
        'wb_buoy'
    ]
        colors = [
        'black',    
        'blue',     
        'orange',   
        'green',    
        'red',      
        'purple',   
        'brown',    
        'pink',     
        'grey',
        'cyan'
    ]
    
        for var in vars:
            globals()[f"out_{var}"]=averaged_profiles(var)
    
        # for var in vars:
        #     out_profile=globals()[f"out_{var}"]
        #     minmax=(np.min(out_profile[:,0]),np.max(out_profile[:,0]))
    
        if type=='general':
            axis=ax1
            plotting(out_w,axis,label=f'{type} ' + vars[0],color=colors[0])
            axis.set_ylim(bottom=0,top=20)
            axis.set_xlabel('w (m/s)')
            axis.legend()
            axis=ax3
        if type=='cloudy':
            axis=ax2
            plotting(out_w,axis,label=f'{type} ' + vars[0],color=colors[1])
            axis.set_ylim(bottom=0,top=20)
            axis.set_xlabel('w (m/s)')
            axis.legend()
            axis=ax4
        for var,xlabel,color in zip(vars[1:],vars[1:],colors[1:]):
            plotting(globals()[f"out_{var}"],axis,label=xlabel,color=color)
            axis.set_ylim(bottom=0,top=20)
            axis.set_ylabel('z (km)');
            axis.set_xlabel('(m/s/s)')
            axis.legend(fontsize='small'); #add legend
    
    #FIXING X LIMITS
    fix_x_limits([ax1, ax2])
    fix_x_limits([ax3, ax4])
    
    #MEAN CLOUD BASE
    axises=[ax1,ax2,ax3,ax4]
    w_profile=averaged_profiles('w')
    cloudbase=w_profile[0,1]
    for axis in axises:
        axis.axhline(cloudbase,color='purple',linestyle='dashed',linewidth=1)
    
    #save plot
    if dim=='1km':
        plt.savefig(dir+f'Project_Algorithms/plots/1km_eulerian_generalvscloudy_updraft_WBUDGET_profile_wthresh{label}.jpg', bbox_inches='tight', dpi=300) 
    # if dim=='250m'

In [44]:
if plotting2==True:
    #QV BUDGETS
    from matplotlib.ticker import ScalarFormatter
    label=1
    
    import h5py
    #Produced averaged profiles for plotting
    def averaged_profiles(var):
        profile=globals()[f"profile_{var}"]
    
        out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
        out_var=np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T #divides the data column by the counter column
        return out_var
    
    ####################################################################
    dim='1km'
    label=1
    
    #Uses Plotting Function 
    fig, axs = plt.subplots(2, 2, figsize=(10, 10))
    plt.subplots_adjust(top=0.95)
    
    fig.suptitle(f"Average General (" + r"$w_{thresh}$: w>= " + f"{w_thresh1} & " + r"$(q_c+q_i)_{thresh}$ < " + f"{qcqi_thresh}" + ") vs Cloudy (" + r"$w_{thresh}$: w>= " + f"{w_thresh2} & " + r"$(q_c+q_i)_{thresh}$ >= " + f"{qcqi_thresh}" + ") Updraft QVBUDGET Profiles (Eulerian)")
    ax1, ax2, ax3, ax4 = axs.flatten()
    
    ####################################################################
    for type in ["general","cloudy"]:
        print(f'currently on type {type}')
        
        if type == "general":
            input_file = dir2+f'job_out/QV_BUDGET_general_eulerian_profiles_{res}_{Np_str}_5min.h5' 
        elif type == "cloudy":
            input_file = dir2+f'job_out/QV_BUDGET_cloudy_eulerian_profiles_{res}_{Np_str}_5min.h5'
        
        with h5py.File(input_file, 'r') as f:
            profile_qv = np.array(f['profile_qv'])  
            profile_qvb_hadv = np.array(f['profile_qvb_hadv'])  
            profile_qvb_vadv = np.array(f['profile_qvb_vadv'])  
            profile_qvb_hidiff = np.array(f['profile_qvb_hidiff']) 
            profile_qvb_vidiff = np.array(f['profile_qvb_vidiff'])   
            profile_qvb_hturb = np.array(f['profile_qvb_hturb'])  
            profile_qvb_vturb = np.array(f['profile_qvb_vturb'])  
            profile_qvb_mp = np.array(f['profile_qvb_mp']) 
    
        #Plotting function
        def plotting(out_var,axis,label,color):
            axis.plot(out_var[:,0],out_var[:,1],label=label,color=color)
            axis.grid(True)
        
        #Uses Averaged_Profiles Function
        vars = [
            'qv',
            'qvb_hadv',
            'qvb_vadv',
            'qvb_hidiff',
            'qvb_vidiff',
            'qvb_hturb',
            'qvb_vturb',
            'qvb_mp'
        ]
        
        colors = [
        'black',    
        'blue',     
        'orange',   
        'green',    
        'red',      
        'purple',   
        'brown',    
        'pink',     
        'grey',
        'cyan'
    ]
    
        for var in vars:
            globals()[f"out_{var}"]=averaged_profiles(var)
            globals()[f"out_{var}"][:,0]*=1000
    
        if type=='general':
            axis=ax1
            plotting(out_qv,axis,label=f'{type} ' + vars[0],color=colors[0])
            axis.set_ylim(bottom=0)
            axis.set_xlabel('qv (g/kg)')
            axis.legend()
            axis=ax3
        if type=='cloudy':
            axis=ax2
            plotting(out_qv,axis,label=f'{type} ' + vars[0],color=colors[1])
            axis.set_ylim(bottom=0)
            axis.set_xlabel('qv (g/kg)')
            axis.legend()
            axis=ax4
        for var,xlabel,color in zip(vars[1:],vars[1:],colors[1:]):
            plotting(globals()[f"out_{var}"],axis,label=xlabel,color=color)
            axis.set_ylabel('z (km)');
            axis.set_xlabel('(g/kg/s)')
            axis.legend(fontsize='small'); #add legend
            axis.xaxis.set_major_formatter(ScalarFormatter())
            axis.xaxis.get_major_formatter().set_powerlimits((-1, 1))  # Set limits for scientific notation
    
    #FIXING X LIMITS
    fix_x_limits([ax1, ax2])
    fix_x_limits([ax3, ax4])
    
    #MEAN CLOUD BASE
    axises=[ax1,ax2,ax3,ax4]
    w_profile=averaged_profiles('w')
    cloudbase=w_profile[0,1]
    for axis in axises:
        axis.axhline(cloudbase,color='purple',linestyle='dashed',linewidth=1)
    
                
    #save plot
    if dim=='1km':
        plt.savefig(dir+f'Project_Algorithms/plots/1km_eulerian_generalvscloudy_updraft_QVBUDGET_profile_wthresh{label}.jpg', bbox_inches='tight', dpi=300) 
    #limited

In [45]:
if plotting2==True:
    #TH BUDGETS
    from matplotlib.ticker import ScalarFormatter
    label=1
    
    import h5py
    #Produced averaged profiles for plotting
    def averaged_profiles(var):
        profile=globals()[f"profile_{var}"]
    
        out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
        out_var=np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T #divides the data column by the counter column
        return out_var
    
    ####################################################################
    dim='1km'
    label=1
    
    #Uses Plotting Function 
    fig, axs = plt.subplots(2, 2, figsize=(10, 10))
    plt.subplots_adjust(top=0.95)
    
    fig.suptitle(f"Average General (" + r"$w_{thresh}$: w>= " + f"{w_thresh1} & " + r"$(q_c+q_i)_{thresh}$ < " + f"{qcqi_thresh}" + ") vs Cloudy (" + r"$w_{thresh}$: w>= " + f"{w_thresh2} & " + r"$(q_c+q_i)_{thresh}$ >= " + f"{qcqi_thresh}" + ") Updraft THBUDGET Profiles (Eulerian)")
    ax1, ax2, ax3, ax4 = axs.flatten()
    
    ####################################################################
    for type in ["general","cloudy"]:
        print(f'currently on type {type}')
        
        if type == "general":
            input_file = dir2+f'job_out/TH_BUDGET_general_eulerian_profiles_{res}_{Np_str}_5min.h5' 
        elif type == "cloudy":
            input_file = dir2+f'job_out/TH_BUDGET_cloudy_eulerian_profiles_{res}_{Np_str}_5min.h5' 
        
        with h5py.File(input_file, 'r') as f:
            profile_th = np.array(f['profile_th'])  
            profile_ptb_hadv = np.array(f['profile_ptb_hadv'])  
            profile_ptb_vadv = np.array(f['profile_ptb_vadv'])  
            profile_ptb_hidiff = np.array(f['profile_ptb_hidiff']) 
            profile_ptb_vidiff = np.array(f['profile_ptb_vidiff'])  
            profile_ptb_hturb = np.array(f['profile_ptb_hturb'])  
            profile_ptb_vturb = np.array(f['profile_ptb_vturb'])  
            profile_ptb_mp = np.array(f['profile_ptb_mp']) 
            profile_ptb_rad = np.array(f['profile_ptb_rad']) 
            profile_ptb_div = np.array(f['profile_ptb_div']) 
            profile_ptb_diss = np.array(f['profile_ptb_diss']) 
    
        #Plotting function
        def plotting(out_var,axis,label,color):
            axis.plot(out_var[:,0],out_var[:,1],label=label,color=color)
            axis.grid(True)
        
        #Uses Averaged_Profiles Function
        vars = [
            'th',
            'ptb_hadv',
            'ptb_vadv',
            'ptb_hidiff',
            'ptb_vidiff',
            'ptb_hturb',
            'ptb_vturb',
            'ptb_mp',
            'ptb_rad',
            'ptb_div',
            'ptb_diss'
        ]
        
        colors = [
        'black',    
        'blue',     
        'orange',   
        'green',    
        'red',      
        'purple',   
        'brown',    
        'pink',     
        'grey',
        'cyan'
    ]
    
        for var in vars:
            globals()[f"out_{var}"]=averaged_profiles(var)
    
        if type=='general':
            axis=ax1
            plotting(out_th,axis,label=f'{type} ' + vars[0],color=colors[0])
            axis.set_ylim(bottom=0)
            axis.set_xlabel('qv (g/kg)')
            axis.legend()
            axis=ax3
        if type=='cloudy':
            axis=ax2
            plotting(out_th,axis,label=f'{type} ' + vars[0],color=colors[1])
            axis.set_ylim(bottom=0)
            axis.set_xlabel('qv (g/kg)')
            axis.legend()
            axis=ax4
        for var,xlabel,color in zip(vars[1:],vars[1:],colors[1:]):
            plotting(globals()[f"out_{var}"],axis,label=xlabel,color=color)
            axis.set_ylabel('z (km)');
            axis.set_xlabel('(g/kg/s)')
            axis.legend(fontsize='small'); #add legend
            axis.xaxis.set_major_formatter(ScalarFormatter())
            axis.xaxis.get_major_formatter().set_powerlimits((-1, 1))  # Set limits for scientific notation
    
    #FIXING X LIMITS
    fix_x_limits([ax1, ax2])
    fix_x_limits([ax3, ax4])
    
    #MEAN CLOUD BASE
    axises=[ax1,ax2,ax3,ax4]
    w_profile=averaged_profiles('w')
    cloudbase=w_profile[0,1]
    for axis in axises:
        axis.axhline(cloudbase,color='purple',linestyle='dashed',linewidth=1)
    
                
    #save plot
    if dim=='1km':
        plt.savefig(dir+f'Project_Algorithms/plots/1km_eulerian_generalvscloudy_updraft_THBUDGET_profile_wthresh{label}.jpg', bbox_inches='tight', dpi=300) 
    #limited